# Aula 04: Análise Exploratória de Dados (EDA) e Regras de Negócio
**Capacitação de Python para Ciência de Dados — Case Campe Supply**

Agora que nossa base está limpa, validada e sem outliers mentirosos, vamos atuar como analistas e cientistas de dados para responder perguntas estratégicas da diretoria da **Campe Supply**:
1. Qual é o faturamento total, lucro total e ticket médio da empresa?
2. Quem são os melhores vendedores em **Faturamento** vs. **Lucro Líquido**?
3. Qual **Setor** e **Produto** trazem o maior retorno financeiro?
4. Como as vendas se comportam entre **Atacado, Atacarejo e Varejo** em cada Região (`pivot_table`)?

In [ ]:
import pandas as pd
import numpy as np
from difflib import get_close_matches

# Carregando e executando o pipeline completo de limpeza (caso o aluno rode direto a Aula 04)
df = pd.read_excel('Campe_Supply_Corrompido.xlsx', sheet_name='Campe Supply')
df = df.drop(columns=['UNNAMED: 0', 'coluna_inutil_0', 'coluna_inutil_1'])
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('ã', 'a')
df = df.drop_duplicates().reset_index(drop=True)

categorias_oficiais = {
    'setor': ['Congelados', 'Latícinio', 'Bebidas', 'Carnes'],
    'produto': ['Queijo', 'Sorvete', 'Frango', 'Refrigerantes', 'Cervejas', 'Bovina', 'Leite', 'Lasanha', 'Pizza', 'Suína', 'Sucos', 'Coalhada'],
    'vendedor': ['Vanessa', 'Amaral', 'Leo', 'Daniele', 'Jonas', 'Evandro'],
    'regiao': ['Sudeste', 'Nordeste', 'Sul'],
    'estado': ['Rio de Janeiro', 'Minas Gerais', 'Bahia', 'Rio Grande do Norte', 'São Paulo', 'Sergipe', 'Rio Grande do Sul', 'Paraná', 'Santa Catarina'],
    'cliente': ['Atacarejo', 'Varejo', 'Atacado']
}
def corrigir_palavra(valor, lista_oficial):
    if pd.isna(valor): return np.nan
    match = get_close_matches(str(valor).strip(), lista_oficial, n=1, cutoff=0.6)
    return match[0] if match else np.nan

for col, lista_valida in categorias_oficiais.items():
    df[col] = df[col].apply(lambda x: corrigir_palavra(x, lista_valida))

mapa_produto_setor = {'Queijo': 'Latícinio', 'Leite': 'Latícinio', 'Coalhada': 'Latícinio', 'Sorvete': 'Congelados', 'Lasanha': 'Congelados', 'Pizza': 'Congelados', 'Frango': 'Carnes', 'Bovina': 'Carnes', 'Suína': 'Carnes', 'Refrigerantes': 'Bebidas', 'Cervejas': 'Bebidas', 'Sucos': 'Bebidas'}
mapa_estado_regiao = {'Rio de Janeiro': 'Sudeste', 'Minas Gerais': 'Sudeste', 'São Paulo': 'Sudeste', 'Bahia': 'Nordeste', 'Rio Grande do Norte': 'Nordeste', 'Sergipe': 'Nordeste', 'Rio Grande do Sul': 'Sul', 'Paraná': 'Sul', 'Santa Catarina': 'Sul'}
df['setor'] = df['setor'].fillna(df['produto'].map(mapa_produto_setor))
df['regiao'] = df['regiao'].fillna(df['estado'].map(mapa_estado_regiao))

for col in ['faturamento', 'margem_de_lucro', 'lucro']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.loc[df['lucro'] > df['faturamento'], 'lucro'] = df['faturamento'] * df['margem_de_lucro']
df.loc[(df['margem_de_lucro'] == 0) & (df['lucro'] > 0), 'margem_de_lucro'] = df['lucro'] / df['faturamento']
df.loc[(df['faturamento'] * df['margem_de_lucro'] - df['lucro']).abs() > 0.01, 'faturamento'] = df['lucro'] / df['margem_de_lucro']

df['lucro'] = df['lucro'].fillna(df['faturamento'] * df['margem_de_lucro']).round(2)
df['faturamento'] = df['faturamento'].fillna(df['lucro'] / df['margem_de_lucro']).round(2)
df['margem_de_lucro'] = df['margem_de_lucro'].fillna(df['lucro'] / df['faturamento']).round(2)
df = df.drop_duplicates().reset_index(drop=True)
df['mes'] = df['data_da_venda'].dt.month

## 1. KPIs Gerais da Empresa
Vamos calcular os grandes números da operação da Campe Supply:

In [ ]:
faturamento_total = df['faturamento'].sum()
lucro_total = df['lucro'].sum()
ticket_medio = df['faturamento'].mean()
margem_media = (df['margem_de_lucro'].mean()) * 100

print(f'Faturamento Total: R$ {faturamento_total:,.2f}')
print(f'Lucro Líquido Total: R$ {lucro_total:,.2f}')
print(f'Ticket Médio por Venda: R$ {ticket_medio:,.2f}')
print(f'Margem de Lucro Média: {margem_media:.2f}%')

## 2. Ranking de Vendedores (`groupby` + `agg`)
Como avaliamos nossa equipe comercial? Vamos agrupar por `vendedor` e calcular quantidade de vendas, faturamento total, lucro total, ticket médio e margem média.

In [ ]:
ranking_vendedores = (
    df.groupby('vendedor')
    .agg(
        qtd_vendas=('faturamento', 'count'),
        faturamento_total=('faturamento', 'sum'),
        lucro_total=('lucro', 'sum'),
        ticket_medio=('faturamento', 'mean'),
        margem_media=('margem_de_lucro', 'mean')
    )
    .sort_values(by='lucro_total', ascending=False)
    .round(2)
)
ranking_vendedores

## 3. Desempenho por Setor e Produto
Quais setores e produtos são o carro-chefe da Campe Supply?

In [ ]:
analise_setor = (
    df.groupby('setor')
    .agg(
        qtd_vendas=('faturamento', 'count'),
        faturamento_total=('faturamento', 'sum'),
        lucro_total=('lucro', 'sum'),
        ticket_medio=('faturamento', 'mean')
    )
    .sort_values(by='faturamento_total', ascending=False)
    .round(2)
)
analise_setor

In [ ]:
# Top 10 Produtos por Lucro Total
top_produtos = (
    df.groupby(['setor', 'produto'])
    .agg(
        qtd_vendas=('faturamento', 'count'),
        faturamento_total=('faturamento', 'sum'),
        lucro_total=('lucro', 'sum')
    )
    .sort_values(by='lucro_total', ascending=False)
    .round(2)
)
top_produtos

## 4. Tabela Dinâmica (`pivot_table`): Região vs. Tipo de Cliente
Para cruzar duas variáveis categóricas e ver uma métrica financeira no meio (igual à Tabela Dinâmica do Excel), usamos `pd.pivot_table()`.

In [ ]:
tabela_regiao_cliente = pd.pivot_table(
    df,
    values='faturamento',
    index='regiao',
    columns='cliente',
    aggfunc='sum',
    margins=True,
    margins_name='Total Geral'
).round(2)

tabela_regiao_cliente

## 5. Filtros Avançados com `.query()`
E se o diretor comercial pedir apenas as vendas de **Alto Valor** (`faturamento > 1500`) feitas na região **Sudeste**?

In [ ]:
vendas_vip_sudeste = df.query("regiao == 'Sudeste' and faturamento > 1500").sort_values(by='faturamento', ascending=False)
print(f'Total de vendas VIP no Sudeste: {len(vendas_vip_sudeste)}')
vendas_vip_sudeste.head(10)